#### Setup

We now proceed with analysis of the accumulated and cleaned data.

In [ ]:
import pandas as pd
import numpy as np
import pyarrow
import os

DATA_PATH = "C:/Dev/smard-merit-order/data"

# NOTE: merged used here is merged_v2.xlsx from graphing.ipynb
merged = pd.read_parquet(os.path.join(DATA_PATH, "merged_v2.parquet"))

merged['renewable_penetration'] = merged['total_renewables'] / merged['gl']
merged['hour'] = merged['timestamp'].dt.hour
merged['month'] = merged['timestamp'].dt.month
merged['day_of_week'] = merged['timestamp'].dt.dayofweek
merged['crisis_dummy'] = (merged['period'] == 'Crisis').astype(int)

print(merged[['renewable_penetration', 'hour', 'month', 'day_of_week', 'crisis_dummy']].describe())
print(merged['crisis_dummy'].value_counts())

In [ ]:
import statsmodels.api as sm
import statsmodels.formula.api as smf
from statsmodels.stats.outliers_influence \
import variance_inflation_factor as VIF
from statsmodels.stats.anova import anova_lm

# naive model
model0 = smf.ols(formula='dap ~ total_renewables', data=merged)
res0 = model0.fit()
print(res0.summary())

# base model
model1 = smf.ols(formula='dap ~ total_renewables + gl + C(hour) + C(month) + C(day_of_week) + crisis_dummy', data=merged)
res1 = model1.fit()
print(res1.summary())

From the results we can clearly see that the model suffers from heteroskedasticity, autocorrelation, and multicollinearity. This is to be expected as the prices at each hour are related to those nearby. For example, the price at 3PM is correlated with that of 2PM. Multicollinearity arises from the fact that the predictors are highly correlated with each other. For example, gl(gridload) and total_renewables are correlated because both peak during daytime.

Now we use HAC to correct this.

In [ ]:
res1_1 = model1.fit(cov_type='HAC', cov_kwds={'maxlags': 24}) # 24 for hours in a day
print(res1_1.summary())

Now the correlation coefficients to decide which regressors to drop and replace.

In [ ]:
print(merged[['total_renewables', 'gl', 'renewable_penetration']].corr())

`total_renewables` is highly correlated with `renewable_penetration` and also somewhat with gridload.

In [ ]:
model2 = smf.ols(formula='dap ~ renewable_penetration + gl + C(hour) + C(month) + C(day_of_week) + crisis_dummy', data=merged)
res2 = model2.fit(cov_type='HAC', cov_kwds={'maxlags' : 24})
print(res2.summary())

model3 = smf.ols(formula='dap ~ renewable_penetration + C(hour) + C(month) + C(day_of_week) + crisis_dummy', data=merged)
res3 = model3.fit(cov_type='HAC', cov_kwds={'maxlags' : 24})
print(res3.summary())

`model3` has a condition number of 29.4 versus 1.54e+06 in Model2 that's multicollinearity essentially eliminated by dropping gl. R² only drops from 0.462 to 0.458, meaning gl was consuming a regression slot while explaining almost nothing that hour and month dummies weren't already capturing.

Now we split pre and post crisis models to see whether the relationship changed after the crisis.

In [ ]:
pre = merged[merged['period'] == 'Pre-crisis']
post = merged[merged['period'] == 'Post-crisis']

model_pre = smf.ols(formula='dap ~ renewable_penetration + C(hour) + C(month) + C(day_of_week)', data=pre)
res_pre = model_pre.fit(cov_type='HAC', cov_kwds={'maxlags' : 24})
print(res_pre.summary())

model_post = smf.ols(formula='dap ~ renewable_penetration + C(hour) + C(month) + C(day_of_week)', data=post)
res_post = model_post.fit(cov_type='HAC', cov_kwds={'maxlags' : 24})
print(res_post.summary())

The merit order effect more than doubled in strength. One unit increase in renewable penetration suppressed price by €66 pre-crisis and €172 post-crisis. That's a 161% increase in the price-suppressing effect of renewables. $R^2$ also improved (60.6 and 59.8) vs the previos 46.2. This shows that there are two totally different relations before and after the crisis.

### Logistic Regression

Currently `is_negative` is either 1 or 0, if we run an OLS on it, we'll get numbers like 1.3 or -0.7 which make no sense. Also, OLS assumes the relationship between predictors and outcome is linear all the way through — but the effect of renewable penetration on the probability of a negative price isn't linear. Going from penetration 0.2 to 0.3 probably has a small effect. Going from 0.8 to 0.9 probably has a large effect because you're approaching the critical threshold. OLS has no way to represent this. 

Logistic regression predits the *probability* that the outcome is 1. This guarantees that the output will be between 0 and 1.

Note: In logistic regression, the coefficients are *log-odds* not simple linear coefficients. To convert them into marginal effects, statsmodels has `.get_margeff()` method for this.

In [ ]:
merged['is_negative'] = merged['is_negative'].astype(int)
logit_model = smf.logit(formula='is_negative ~ renewable_penetration + C(hour) + C(month) + C(day_of_week)', data=merged)
logit_res = logit_model.fit()
print(logit_res.summary())

print(logit_res.get_margeff().summary())

### Gradient Boosting Model


In [ ]:
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.metrics import classification_report
from sklearn.linear_model import LogisticRegression
import matplotlib.pyplot as plt

FEATURES = ['renewable_penetration', 'hour', 'month', 'day_of_week', 'slr', 'won', 'wof']
TARGET = 'is_negative'

train = merged[merged['timestamp'] < '2025-01-01']
test  = merged[merged['timestamp'] >= '2025-01-01']

X_train = train[FEATURES]
y_train = train[TARGET]

X_test = test[FEATURES]
y_test = test[TARGET]

print(f"Training hours: {len(train)}")
print(f"Test hours: {len(test)}")
print(f"Negative hours in test: {y_test.sum()} ({100*y_test.mean():.1f}%)")

lr = LogisticRegression(max_iter=1000)
lr.fit(X_train, y_train)
lr_preds = lr.predict(X_test)

print("LOGISTIC REGRESSION BASELINE")
print(classification_report(y_test, lr_preds))

gb = GradientBoostingClassifier(
    n_estimators=200,
    max_depth=4,
    learning_rate=0.05,
    random_state=42
)
gb.fit(X_train, y_train)
gb_preds = gb.predict(X_test)

print("GRADIENT BOOSTING")
print(classification_report(y_test, gb_preds))

importances = gb.feature_importances_
feat_df = pd.DataFrame({
    'feature': FEATURES,
    'importance': importances
}).sort_values('importance', ascending=True)

feat_df.plot.barh(x='feature', y='importance', title='Feature Importances')
plt.tight_layout()
plt.show()